In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib import lines as mlines
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.kernel_ridge import KernelRidge
from sklearn.metrics import mean_absolute_error
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor

from sklearn.kernel_ridge import KernelRidge

import helpers as hp

### Pre-Process Modeling Data

In [ ]:
df = pd.read_excel('fluoride_modeling_properties_w_experimental_data.xlsx',
                   sheet_name='CS2_combined_ddg', header=1)

df_train = df[df['set'] == 'train']
df_internal_test = df[df['set'] == 'validation']
df_external_test = df[df['set'] == 'test']

# train
df_exp_train = df_train.loc[:,:'ligand_class']
df_features_train = df_train.loc[:,'HOMO_Boltz':]

# validation
df_exp_internal_test = df_internal_test.loc[:,:'ligand_class']
df_features_internal_test = df_internal_test.loc[:,'HOMO_Boltz':]

# test
df_exp_external_test = df_external_test.loc[:,:'ligand_class']
df_features_external_test = df_external_test.loc[:,'HOMO_Boltz':]

#### Remove Collinear Features

This is an optional step that was evaluated for model performance but not used to produce the final model 

In [ ]:
threshold = 0.7

# remove collinear features 
print(f'Shape of descriptors file before removing parameters with R^2 > {threshold} :',df_features_train.shape)
df_corr = df_features_train.corr()
df_not_correlated = ~(df_corr.mask(np.tril(np.ones([len(df_corr)]*2, dtype=bool))).abs() > threshold).any()
un_corr_idx = df_not_correlated.loc[df_not_correlated[df_not_correlated.index] == True].index
df_features_train = df_features_train[un_corr_idx]
df_features_internal_test = df_features_internal_test[un_corr_idx]
df_features_external_test = df_features_external_test[un_corr_idx]
print(f'Shape of descriptors file after removing parameters with R^2 > {threshold} : ',df_features_train.shape)


#### Scale Modeling Data

all features were scaled using the standard scaler for all modeling efforts

In [ ]:
scaler = StandardScaler()
df_features_train_scaled = pd.DataFrame(scaler.fit_transform(df_features_train), columns=df_features_train.columns)
df_features_internal_test_scaled = pd.DataFrame(scaler.transform(df_features_internal_test), columns=df_features_internal_test.columns)
df_features_external_test_scaled = pd.DataFrame(scaler.transform(df_features_external_test), columns=df_features_external_test.columns)

train_scaled = pd.concat([df_exp_train.reset_index(drop=True), df_features_train_scaled.reset_index(drop=True)], axis=1)
internal_test_scaled = pd.concat([df_exp_internal_test.reset_index(drop=True), df_features_internal_test_scaled.reset_index(drop=True)], axis=1)
external_test_scaled = pd.concat([df_exp_external_test.reset_index(drop=True), df_features_external_test_scaled.reset_index(drop=True)], axis=1)

df_scaled = pd.concat([train_scaled, internal_test_scaled, external_test_scaled], axis=0).reset_index(drop=True)

#### Final processing for modeling

In [ ]:
X_train = df_features_train_scaled
y_train = (train_scaled['ddg_flipped'])
y_train_labels = train_scaled['ligandID']

X_internal_test = df_features_internal_test_scaled
y_internal_test = (internal_test_scaled['ddg_flipped'])
y_internal_test_labels = internal_test_scaled['ligandID']

X_external_test = df_features_external_test_scaled
y_external_test = (external_test_scaled['ddg_flipped'])
y_external_test_labels = external_test_scaled['ligandID']

### KNN-R Modeling 

#### Tune hyperparameters to find best model 

In [ ]:
best_df_r2, best_params_r2, best_features_r2, best_df_mae, best_params_mae, best_features_mae = hp.model_selection_knn(
    X_train, y_train, y_train_labels,
    X_internal_test, y_internal_test, y_internal_test_labels,
    scoring_list=[f_regression, mutual_info_regression],
    k_list=[2, 3, 4, 5], #limit to what is reasonable for this size dataset
    neighbors_list=[3, 5, 10])

#### Look at one Model

In [ ]:
# perform feature selection
X_train_selected, X_internal_test_selected, X_external_test_selected, selected_features = hp.feature_selection(
    selection_method='select_k_best', #choose selection method, here we used k best
    n_features=3, # choose number of features you want in model 
    X_train=X_train,
    y_train=y_train,
    scoring=f_regression,
    X_internal_test=X_internal_test,
    X_external_test=X_external_test
)

In [ ]:
# model and plot results using knn
knn_results = hp.regression('knn-r', X_train_selected, y_train, y_train_labels,
               n_neighbors=10, weights='distance', p=1,
               X_internal_test=X_internal_test_selected, y_internal_test=y_internal_test, 
               y_internal_test_labels=y_internal_test_labels,
               X_external_test=X_external_test_selected, y_external_test=y_external_test,
               y_external_test_labels=y_external_test_labels)

# plot results 
fig, ax = plt.subplots(figsize=(6, 6))
ax.set_xlabel('Experimental $ΔΔG^{‡}$', fontsize=16)
ax.set_ylabel('Predicted $ΔΔG^{‡}$ (kcal/mol)', fontsize=16)
subset_train = knn_results[knn_results['set'] == 'train']
ax.scatter(subset_train['true_value'], subset_train['predicted_value_knn'], color='black', label='Train', alpha=0.7)
subset_internal_test = knn_results[knn_results['set'] == 'internal_test']
ax.scatter(subset_internal_test['true_value'], subset_internal_test['predicted_value_knn'], color='blue', label='Internal Test', alpha=0.7)
subset_external_test = knn_results[knn_results['set'] == 'external_test']
ax.scatter(subset_external_test['true_value'], subset_external_test['predicted_value_knn'], color='green', label='External Test', alpha=0.7)
ax.legend()

### KRR + Hyperparameter Tuning

#### Tune hyperparameters to find best model

In [ ]:
best_df_r2, best_params_r2, best_features_r2, best_df_mae, best_params_mae, best_features_mae =  hp.model_selection_krr(X_train, y_train, y_train_labels,
                        X_internal_test, y_internal_test, y_internal_test_labels,
                        ridge_alpha_list=[0.1, 0.3, 1.0],
                        krr_alpha_list=[0.3, 0.7, 1.0],
                        kernel_list=['linear', 'polynomial', 'rbf'],
                        n_features_list=[2, 3, 4, 5])

#### Try One Model 

##### CV To Select Optimal Number of Features for You

In [ ]:
X_train_selected, X_internal_test_selected, X_external_test_selected, selected_features = hp.feature_selection(
    selection_method='rfe_cv',   # or 'rfe_cv'
    X_train=X_train,
    y_train=y_train,
    alpha=0.3,
    min_features=2,
    scoring = 'neg_mean_squared_error',
    X_internal_test=X_internal_test,
    X_external_test=X_external_test
)

##### Select Optimal Number of Features for RFE yourself

In [ ]:
X_train_selected, X_internal_test_selected, X_external_test_selected, selected_fearures = hp.feature_selection(
    selection_method='rfe', 
    X_train=X_train,
    y_train=y_train,
    alpha=0.3,
    n_features = 4,
    X_internal_test=X_internal_test,
    X_external_test=X_external_test
)

##### Try KRR Modeling 

In [ ]:
krr_results = hp.regression('krr', X_train_selected, y_train, y_train_labels,
               alpha = 0.3,
               kernel='polynomial',
               X_internal_test=X_internal_test_selected, y_internal_test=y_internal_test, 
               y_internal_test_labels=y_internal_test_labels,
               X_external_test=X_external_test_selected, y_external_test=y_external_test,
               y_external_test_labels=y_external_test_labels)

fig, ax = plt.subplots(figsize=(6, 6))
ax.set_xlabel('Experimental $ΔΔG^{‡}$', fontsize=16)
ax.set_ylabel('Predicted $ΔΔG^{‡}$ (kcal/mol)', fontsize=16)
subset_train = krr_results[krr_results['set'] == 'train']
ax.scatter(subset_train['true_value'], subset_train['predicted_value_krr'], color='black', label='Train', alpha=0.7)
subset_internal_test = krr_results[krr_results['set'] == 'internal_test']
ax.scatter(subset_internal_test['true_value'], subset_internal_test['predicted_value_krr'], color='blue', label='Internal Test', alpha=0.7)
subset_external_test = krr_results[krr_results['set'] == 'external_test']
ax.scatter(subset_external_test['true_value'], subset_external_test['predicted_value_krr'], color='green', label='External Test', alpha=0.7)
ax.legend()